In [ ]:
# Cell 1: Imports & Autoreload
%load_ext autoreload
%autoreload 2

In [4]:

import sys
import os
import pandas as pd
sys.path.append(os.path.abspath("../"))

from src.eda_utilities import  load_data
from src.embedding import (
    stratified_sample, chunk_dataframe, load_embedding_model,
    embed_chunks, build_chroma_store, load_chroma_store, query_store
)

print("Vector Engine successfully hooked to notebook!")

Vector Engine successfully hooked to notebook!


In [5]:
# Shrink the massive dataset down to 12,000 proportionally representative rows
print("Creating stratified sample...")

# Assuming your cleaned dataframe from Task 1 is called 'df_cleaned'
# Make sure 'category_col' matches your actual product column name (e.g., 'Product')
root_folder = os.path.abspath("../")
output_path = os.path.join(root_folder, "data","processed","complaintsCleaned.csv")

df_cleaned = load_data(output_path)
df_sample = stratified_sample(df_cleaned, n=12000, category_col='Product')

print(f"Sample created with {len(df_sample)} rows.")
print(df_sample['Product'].value_counts(normalize=True) * 100)

Creating stratified sample...
Loading data from: c:\Users\Hermela\Desktop\10academy\Week7\data\processed\complaintsCleaned.csv
Shape  : 454,468 rows × 18 columns
Memory : 759.9 MB
Sample created with 12000 rows.
Product
Credit Card        41.658333
Savings Account    30.875000
Money Transfer     21.716667
Personal Loan       5.750000
Name: proportion, dtype: float64


In [7]:
# Chop the 12,000 complaints into 500-character pieces with metadata attached
print("Chunking text narratives...")

chunks_list = chunk_dataframe(df_sample, text_col="Consumer complaint narrative")

print(f"Total chunks generated: {len(chunks_list)}")
print("\nPreview of Chunk 0:")
print(chunks_list[0])

Chunking text narratives...
Chunking narratives...


Splitting text: 100%|██████████| 12000/12000 [00:05<00:00, 2079.76it/s]

Total chunks generated: 22205

Preview of Chunk 0:
{'text': 'credit card jp morgan chase one day suddenly find charge two transaction one 1500 00 one 500 00 call say balance transfer offer without permission chase initial unauthorized balance transfer charge 4 transaction fee ask reverse transations operator keep say need need anything never authorize transaction chase suspicious make money client', 'metadata': {'Date received': '2022-05-23', 'Product': 'Credit Card', 'Sub-product': 'General-purpose credit card or charge card', 'Issue': 'Other features, terms, or problems', 'Sub-issue': 'Problem with balance transfer', 'Company public response': 'nan', 'Company': 'JPMORGAN CHASE & CO.', 'State': 'NY', 'ZIP code': '11375', 'Tags': 'nan', 'Consumer consent provided?': 'Consent provided', 'Submitted via': 'Web', 'Date sent to company': '2022-05-23', 'Company response to consumer': 'Closed with monetary relief', 'Timely response?': 'Yes', 'Consumer disputed?': 'nan', 'Complaint ID': '55928

In [8]:
print("Loading MiniLM Embedding Model...")
embedding_model = load_embedding_model()

print("Generating vectors (Watch the progress bar)...")
# This creates the massive mathematical array
chunk_embeddings = embed_chunks(chunks_list, model=embedding_model)

print(f"\nSuccessfully created {chunk_embeddings.shape[0]} embeddings.")
print(f"Vector dimensions: {chunk_embeddings.shape[1]}")

Loading MiniLM Embedding Model...


c:\Users\Hermela\Desktop\10academy\Week7\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hermela\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3209.18it/s]


Generating vectors (Watch the progress bar)...
Translating 22,205 text chunks into vector math...


Batches: 100%|██████████| 347/347 [06:47<00:00,  1.18s/it]



Successfully created 22205 embeddings.
Vector dimensions: 384


In [11]:
print("Building local ChromaDB vector store...")

# Save the text, metadata, and mathematical arrays to your hard drive
my_vector_db = build_chroma_store(chunks=chunks_list, embeddings=chunk_embeddings)

print("✅ Database successfully built and saved to disk!")
print(f"Total documents in database: {my_vector_db.count()}")

Building local ChromaDB vector store...
Writing vectors to local ChromaDB database...


Indexing DB: 100%|██████████| 5/5 [01:26<00:00, 17.32s/it]

✅ Database successfully built and saved to disk!
Total documents in database: 22205


In [12]:
test_question = "What should I do if my bank charged me an unfair overdraft fee?"

print(f"🔍 Searching for: '{test_question}'\n")

results = query_store(
    collection=my_vector_db, 
    question=test_question, 
    model=embedding_model, 
    k=3
)

for i, res in enumerate(results, 1):
    print(f"🏆 MATCH {i} (Distance: {res['distance']:.4f})")
    print(f"Category: {res['metadata'].get('Product', 'Unknown')}")
    print(f"Text: {res['text']}\n")

🔍 Searching for: 'What should I do if my bank charged me an unfair overdraft fee?'

🏆 MATCH 1 (Distance: 0.2033)
Category: Savings Account
Text: overdraft fee incorrectly take

🏆 MATCH 2 (Distance: 0.2559)
Category: Savings Account
Text: ive charge lot overdraft fee thing account money didnt pay charge fee day later

🏆 MATCH 3 (Distance: 0.2708)
Category: Savings Account
Text: charge 390 00 overdraft fee contact customer service ask leniency wait check several day late cause account overdraw addition overdraft 10 00 charge 36 00 overdraft 10 00 truly believe predatory charge someone 36 00 draw 10 00

